# `shaply` - demo on five classifiers

This notebook trains **five very different classifiers** on a *synthetic* dataset
built with `sklearn.datasets.make_classification`, computes SHAP values for each,
and renders **every `shaply` figure** for all of them.

Models:

| Model | Family | SHAP explainer |
| --- | --- | --- |
| RandomForest | bagged trees | `TreeExplainer` |
| XGBoost | gradient boosting | `TreeExplainer` |
| LightGBM | gradient boosting | `TreeExplainer` |
| LogisticRegression | linear | `LinearExplainer` |
| SVM (RBF) | kernel | `KernelExplainer` |

The dataset is designed so we know the ground truth: **5 informative** features,
**2 redundant** (linear combinations of the informative ones), and **3 pure
noise** features. A good explainer should rank `inf_*` high and `noise_*` low.

## 1. Setup

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import shap
import shaply

warnings.filterwarnings("ignore")
rng = np.random.default_rng(7)
print("shaply", shaply.__version__, "| shap", shap.__version__)

## 2. A synthetic dataset with known ground truth

In [ ]:
N_INFORMATIVE, N_REDUNDANT, N_NOISE = 5, 2, 3
N_FEATURES = N_INFORMATIVE + N_REDUNDANT + N_NOISE

X, y = make_classification(
    n_samples=2000,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=N_REDUNDANT,
    n_repeated=0,
    n_classes=2,
    class_sep=1.2,
    flip_y=0.01,
    shuffle=False,      # keep column order: informative | redundant | noise
    random_state=7,
)

feature_names = (
    [f"inf_{i}" for i in range(N_INFORMATIVE)]
    + [f"red_{i}" for i in range(N_REDUNDANT)]
    + [f"noise_{i}" for i in range(N_NOISE)]
)

X = pd.DataFrame(X, columns=feature_names)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

# Standardize once: trees are scale-invariant, but the linear and kernel models
# need it - and it keeps SHAP feature-value colors comparable across models.
scaler = StandardScaler().fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train), columns=feature_names, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=feature_names, index=X_test.index)

X_train.head()

## 3. Train the five models

In [ ]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=7),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.1,
        eval_metric="logloss", random_state=7, verbosity=0,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.1, random_state=7, verbose=-1,
    ),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=7),
    "SVM": SVC(kernel="rbf", C=1.0, probability=True, random_state=7),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name:<20s} test accuracy = {model.score(X_test, y_test):.3f}")

## 4. Compute SHAP values → `shaply.Explanation`

Each explainer returns a `shap.Explanation` (or arrays for the kernel case). We
funnel everything through `shaply.to_explanation`, which normalizes any of these
inputs - selecting the positive class for multi-output tree models - into the
single object every `shaply` plot understands.

In [ ]:
# Subsets kept small enough for interactive plots (and for the slow kernel SVM).
X_explain = X_test.iloc[:200]
X_bg = shap.sample(X_train, 60, random_state=7)      # background for the kernel model
X_svm = X_test.iloc[:60]


def to_shaply(explanation) -> shaply.Explanation:
    # Normalize a shap.Explanation, picking the positive class if multi-output.
    output_index = 1 if np.asarray(explanation.values).ndim == 3 else None
    return shaply.to_explanation(explanation, output_index=output_index)


explanations: dict[str, shaply.Explanation] = {}

# Tree models: exact TreeExplainer.
for name in ("RandomForest", "XGBoost", "LightGBM"):
    tree_exp = shap.TreeExplainer(models[name])(X_explain)
    explanations[name] = to_shaply(tree_exp)

# Linear model: LinearExplainer with a background distribution.
lin_exp = shap.LinearExplainer(models["LogisticRegression"], X_train)(X_explain)
explanations["LogisticRegression"] = to_shaply(lin_exp)

# Kernel model: model-agnostic KernelExplainer on predict_proba (class 1).
# Depending on the shap version, shap_values returns either a per-class list or a
# 3D array (n_samples, n_features, n_classes); to_explanation handles both.
kernel = shap.KernelExplainer(models["SVM"].predict_proba, X_bg)
svm_raw = kernel.shap_values(X_svm, nsamples=100)
svm_raw = svm_raw[1] if isinstance(svm_raw, list) else np.asarray(svm_raw)
svm_base = float(np.ravel(kernel.expected_value)[1])
explanations["SVM"] = shaply.to_explanation(
    svm_raw,
    base_values=svm_base,
    data=X_svm.to_numpy(),
    feature_names=feature_names,
    output_index=1 if svm_raw.ndim == 3 else None,
)

{name: (e.n_samples, e.n_features) for name, e in explanations.items()}

## 5. Deep dive: RandomForest, one plot at a time

We start with a single model and walk through the seven `shaply` figures, each
with a one-line reminder of what it answers.

In [ ]:
rf = explanations["RandomForest"]

**Global feature importance** - which features matter most on average?

In [ ]:
shaply.bar(rf).show()

**Beeswarm** - the full distribution of each feature's effect, colored by value.

In [ ]:
shaply.beeswarm(rf).show()

**Heatmap** - SHAP values for every instance (columns) and feature (rows).

In [ ]:
shaply.heatmap(rf).show()

**Waterfall** - how one prediction is built from the base value.

In [ ]:
shaply.waterfall(rf, sample_index=0).show()

**Force** - the same single prediction as opposing red/blue forces.

In [ ]:
shaply.force(rf, sample_index=0).show()

**Decision** - cumulative paths from base value to prediction, one line per instance.

In [ ]:
shaply.decision(shaply.to_explanation(rf.values[:120], base_values=rf.base_values[:120], data=rf.data[:120], feature_names=rf.feature_names)).show()

**Dependence (scatter)** - how one feature's value drives its SHAP value.

In [ ]:
shaply.scatter(rf, "inf_0", color_feature="inf_1").show()

**Beeswarm + value ranges** - the beeswarm on the left, the *real* value distribution of each feature (with true min/max) on the right, so the impact and the concrete operating range are read on the same line.

In [ ]:
shaply.beeswarm_ranges(rf).show()

**Dependence + value ranges** - the same dependence scatter as above, framed by both axes' distributions: a gradient box + violin (colored by the feature's own low->high scale) for the real values, and a plain gray violin + box for the SHAP values.

In [ ]:
shaply.scatter_ranges(rf, "inf_0").show()

## 6. All nine figures for every model

The same helper renders the complete set for each of the five classifiers. Note
how the tree models agree closely, the linear model spreads effects across the
correlated `inf_*`/`red_*` features, and the RBF SVM (kernel-approximated) is
noisier but still ranks the informative features first.

In [ ]:
def subsample(expl: shaply.Explanation, n: int) -> shaply.Explanation:
    # Return the first n instances (keeps the decision plot readable).
    return shaply.to_explanation(
        expl.values[:n],
        base_values=expl.base_values[:n],
        data=None if expl.data is None else expl.data[:n],
        feature_names=expl.feature_names,
    )


PLOTS = [
    ("Global importance (bar)", lambda e: shaply.bar(e)),
    ("Beeswarm", lambda e: shaply.beeswarm(e)),
    ("Heatmap", lambda e: shaply.heatmap(e)),
    ("Waterfall - instance 0", lambda e: shaply.waterfall(e, sample_index=0)),
    ("Force - instance 0", lambda e: shaply.force(e, sample_index=0)),
    ("Decision", lambda e: shaply.decision(subsample(e, 120))),
    ("Dependence inf_0 / inf_1", lambda e: shaply.scatter(e, "inf_0", color_feature="inf_1")),
    ("Beeswarm + value ranges", lambda e: shaply.beeswarm_ranges(e)),
    ("Dependence + value ranges (inf_0)", lambda e: shaply.scatter_ranges(e, "inf_0")),
]


def show_all(name: str, expl: shaply.Explanation) -> None:
    for label, builder in PLOTS:
        fig = builder(expl)
        fig.update_layout(title=f"{name} - {label}")
        fig.show()

### XGBoost

In [ ]:
show_all("XGBoost", explanations["XGBoost"])

### LightGBM

In [ ]:
show_all("LightGBM", explanations["LightGBM"])

### LogisticRegression

In [ ]:
show_all("LogisticRegression", explanations["LogisticRegression"])

### SVM

In [ ]:
show_all("SVM", explanations["SVM"])

## 7. Advanced tools - beyond the usual SHAP plots

These `shaply`-only figures are meant to be *acted on*: they cross SHAP values
with the real data to expose tipping points, coupled effects and failure drivers.
Read them as **associational, not causal** - a strong signal is a lead to
investigate, not a proven cause.

**Response curve** - the smoothed effect of a feature with its ±1 std band and the auto-detected *tipping point* where the effect crosses zero.

In [ ]:
shaply.response_curve(rf, "inf_0").show()

**Interaction heatmap** - which features act *together*. Needs SHAP interaction values (here from the RandomForest via `TreeExplainer.shap_interaction_values`).

In [ ]:
rf_inter = np.asarray(shap.TreeExplainer(models["RandomForest"]).shap_interaction_values(X_explain))
if rf_inter.ndim == 4:            # (n, f, f, n_classes) -> positive class
    rf_inter = rf_inter[..., 1]
shaply.interaction_heatmap(rf_inter, feature_names=feature_names).show()

**Error analysis** - what the RandomForest relies on differently when it is wrong: the mean SHAP per feature on correct vs mis-predicted instances.

In [ ]:
y_true_explain = y_test[:len(X_explain)]
y_pred_explain = models["RandomForest"].predict(X_explain)
shaply.error_analysis(rf, y_true=y_true_explain, y_pred=y_pred_explain).show()

**SHAP surface** - mean SHAP of `inf_0` over the `(inf_0, inf_1)` plane: the operating regions where the feature helps or hurts the prediction.

In [ ]:
shaply.shap_surface(rf, "inf_0", "inf_1").show()

**Importance by cohort** - splitting the instances into quantile bins of `inf_1` shows how each feature's importance shifts between operating regimes.

In [ ]:
shaply.importance_by_cohort(rf, by_feature="inf_1").show()

**Feature redundancy** - features with correlated SHAP vectors act in lockstep. The `red_*` features (built from the informative ones) cluster with them; the `noise_*` features sit apart.

In [ ]:
shaply.feature_clustering(rf).show()

**Explanation archetypes** - clustering instances by SHAP profile reveals the model's recurring decision patterns (each row is one archetype's mean SHAP).

In [ ]:
shaply.explanation_archetypes(rf).show()

**Importance with confidence intervals** and **monotonicity check** - how robust the ranking is, and how cleanly each feature's value drives its SHAP value.

In [ ]:
shaply.importance_ci(rf).show()
shaply.monotonicity_check(rf).show()

## 8. Cross-model comparison of global importance

Finally, a single glance comparing how each model ranks the features. The
informative and redundant features should dominate; the `noise_*` features
should stay near zero for every model - a quick sanity check of both the models
and the explanations.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
for name, expl in explanations.items():
    importance = np.abs(expl.values).mean(axis=0)
    fig.add_bar(x=list(expl.feature_names), y=importance, name=name)

fig.update_layout(
    title="Global feature importance across models - mean(|SHAP|)",
    xaxis_title="Feature",
    yaxis_title="mean(|SHAP value|)",
    barmode="group",
    template="plotly_white",
)
fig.show()